# NILM Model Training on Google Colab

Train a Non-Intrusive Load Monitoring (NILM) disaggregation model using the
`underscore-edge-model-pipeline` package. This notebook covers the full workflow: training,
evaluation, ONNX export, and downloading artefacts.

**Runtime:** Select *Runtime > Change runtime type > T4 GPU* before running.

## 1. Install

The repo is private, so clone it with a [personal access token](https://github.com/settings/tokens) stored in Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
import os
IS_HOSTED_COLAB = "COLAB_RELEASE_TAG" in os.environ
print("Hosted Colab" if IS_HOSTED_COLAB else "Local Colab runtime")

In [ ]:
if IS_HOSTED_COLAB:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
else:
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN via: docker run -e GITHUB_TOKEN=ghp_... ..."
        )

!git clone https://{token}@github.com/underscoreHQ/underscore-edge-model-pipeline.git
%pip install underscore-edge-model-pipeline/

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — training will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 3. Train

In [ ]:
import sys
sys.path.append('/content/underscore-edge-model-pipeline/src')

from edge_pipeline.training.trainer import train_model

model, trainer = train_model(
    model_type="tcn",
    window_size=60,
    batch_size=256,
    max_epochs=30,
    learning_rate=1e-3,
    use_amda=True,
    amda_scale=2.5,
    num_workers=2,
    output_dir="outputs",
    use_wandb=False,
)

## 4. Evaluate

In [ ]:
from edge_pipeline.data.datamodule import NilmDataModule
from edge_pipeline.evaluation.evaluate import APPLIANCE_NAMES, evaluate_model

# Set up a datamodule for evaluation (no augmentation)
eval_dm = NilmDataModule(
    window_size=60,
    batch_size=256,
    num_workers=2,
    use_amda=False,
)
eval_dm.setup()

device = "cuda" if torch.cuda.is_available() else "cpu"
results = evaluate_model(model, eval_dm, device=device)

print(f"{'Metric':<12} {'Value':>10}")
print("-" * 24)
for key in ["mae", "mse", "rmse", "r2", "nde"]:
    print(f"{key:<12} {results[key]:>10.4f}")

print(f"\n{'Appliance':<14} {'MAE':>10} {'R2':>10}")
print("-" * 36)
for name in APPLIANCE_NAMES:
    print(f"{name:<14} {results[f'{name}_mae']:>10.4f} {results[f'{name}_r2']:>10.4f}")

## 5. Export to ONNX

In [ ]:
from edge_pipeline.export.onnx_exporter import OnnxExporter

exporter = OnnxExporter(model, output_dir="exports", model_name="nilm_colab")
normalisation_metadata = eval_dm.get_normalisation_metadata()
onnx_path = exporter.export(normalisation_metadata=normalisation_metadata)
print(f"Exported ONNX model to {onnx_path}")

## 6. Download artefacts

In [ ]:
if IS_HOSTED_COLAB:
    from google.colab import files
    files.download(str(onnx_path))
    files.download("exports/nilm_colab_metadata.json")
else:
    print(f"Artefacts saved to:\n  {onnx_path}\n  exports/nilm_colab_metadata.json")
    print("Copy from the container with: docker cp <container>:/content/exports/ ./exports/")